# Week 3 — 06: Reporting

Assemble and save all experiment artefacts (CSVs, model JSON, markdown report).


In [6]:
import sys, pathlib

# Resolve project root so notebooks work regardless of CWD or kernel launch dir
_this_nb  = pathlib.Path(__file__) if "__file__" in dir() else pathlib.Path().resolve()
_sentinel = "src"   # directory that marks the project root
_root     = _this_nb.resolve()
for _ in range(10):                          # walk up at most 10 levels
    if (_root / _sentinel).is_dir():
        break
    _root = _root.parent

# Hard-code fallback for direct kernel launch from notebooks/week3/
_fallback = pathlib.Path("/home/malnaser/Eastnets-Standard-Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project")
if not (_root / _sentinel).is_dir():
    _root = _fallback

if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

PROJECT_ROOT = _root
print("Project root:", PROJECT_ROOT)


Project root: /home/malnaser/Eastnets-Standard-Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project


In [7]:
from src import state
from src.config import ARTIFACTS_DIR, REPORTS_DIR
import numpy as np, pandas as pd
from datetime import datetime

(test_comparison, group_result, test_meta,
 floor_table, OP_THRESHOLD_A, OP_THRESHOLD_B,
 OP_PRECISION_A, OP_RECALL_A, OP_PRECISION_B, OP_RECALL_B,
 pA_test, pB_test, pred_A, pred_B, fusion_table, best_fusion) = state.load("04_eval")

model_A, model_B = state.load("03_models")
(X, feature_cols, train_A_mask, val_A_mask, train_B_mask, val_B_mask,
 X_complete_val, X_complete_test, y_complete_test,
 train_base, val_base, test_base, patterned, unpatterned,
 train_idx, val_idx, test_idx) = state.load("03_matrices")

df                               = state.load("02_df_features")
importance_A, importance_B, comparison = state.load("05_shap")
print("All checkpoints loaded.")


Loaded checkpoint: /home/malnaser/Eastnets-Standard-Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/checkpoints/04_eval.pkl
Loaded checkpoint: /home/malnaser/Eastnets-Standard-Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/checkpoints/03_models.pkl
Loaded checkpoint: /home/malnaser/Eastnets-Standard-Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/checkpoints/03_matrices.pkl
Loaded checkpoint: /home/malnaser/Eastnets-Standard-Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/checkpoints/02_df_features.pkl
Loaded checkpoint: /home/malnaser/Eastnets-Standard-Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/checkpoints/05_shap.pkl
All checkpoints loaded.


In [8]:
# ── Save artefacts ────────────────────────────────────────────────────────────
exp_dir = ARTIFACTS_DIR / "general_plus_pattern_specialist"
exp_dir.mkdir(parents=True, exist_ok=True)

model_A.save_model(exp_dir / "model_A_general_aml.json")
model_B.save_model(exp_dir / "model_B_pattern_specialist.json")
pd.DataFrame({"feature": feature_cols}).to_csv(exp_dir / "feature_columns.csv", index=False)
test_comparison.to_csv(exp_dir / "test_comparison.csv", index=False)
group_result.to_csv(exp_dir / "group_results.csv", index=False)
test_meta.to_csv(exp_dir / "test_predictions.csv", index=False)
floor_table.to_csv(exp_dir / "validation_pr_floors.csv", index=False)
comparison.to_csv(exp_dir / "shap_comparison.csv", index=False)
print("Artefacts saved to:", exp_dir)


Artefacts saved to: /home/malnaser/Eastnets-Standard-Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/artifacts/general_plus_pattern_specialist


In [9]:
# ── Complementarity analysis ─────────────────────────────────────────────────
A_hits = pred_A.astype(bool)
B_hits = pred_B.astype(bool)
Y_pos  = test_meta["target"].to_numpy().astype(bool)

patterned_test   = test_meta[test_meta["target"].eq(1) & test_meta["txt_patterned"].eq(1)]
unpatterned_test = test_meta[test_meta["target"].eq(1) & test_meta["txt_patterned"].eq(0)]

compl = pd.DataFrame({
    "quantity": [
        "All laundering", "Caught by A", "Caught by B", "Caught by both",
        "Caught by A only", "Caught by B only", "Missed by both",
        "Patterned caught by B only", "Unpatterned caught by B only",
    ],
    "count": [
        int(Y_pos.sum()),
        int((Y_pos & A_hits).sum()),
        int((Y_pos & B_hits).sum()),
        int((Y_pos & A_hits & B_hits).sum()),
        int((Y_pos & A_hits & ~B_hits).sum()),
        int((Y_pos & B_hits & ~A_hits).sum()),
        int((Y_pos & ~A_hits & ~B_hits).sum()),
        int((patterned_test["pred_B"].eq(1) & patterned_test["pred_A"].eq(0)).sum()),
        int((unpatterned_test["pred_B"].eq(1) & unpatterned_test["pred_A"].eq(0)).sum()),
    ]
})
print(compl.to_string(index=False))
compl.to_csv(exp_dir / "complementarity.csv", index=False)


                    quantity  count
              All laundering   1265
                 Caught by A    962
                 Caught by B    877
              Caught by both    851
            Caught by A only    111
            Caught by B only     26
              Missed by both    277
  Patterned caught by B only     19
Unpatterned caught by B only      7


In [10]:
# ── Markdown report ───────────────────────────────────────────────────────────
lines = [
    "# AML General Detector + Pattern Specialist — Results",
    "",
    f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "",
    "## Experiment design",
    "",
    "- Model A: normal + ALL laundering transactions.",
    "- Model B: normal + TRUE TXT-patterned laundering; unpatterned excluded from training.",
    "- External Pattern TXT used only for grouping/evaluation — never as model features.",
    "- Leakage-safe chronological split; pattern components kept whole.",
    "",
    "## Dataset",
    "",
    f"- Total rows: **{len(df):,}**",
    f"- Positives: **{int(df.target.sum()):,}**",
    "",
    "## Validation — 15% precision floor",
    "",
    f"- Model A | threshold={OP_THRESHOLD_A:.8f} | precision={OP_PRECISION_A:.4f} | recall={OP_RECALL_A:.4f}",
    f"- Model B | threshold={OP_THRESHOLD_B:.8f} | precision={OP_PRECISION_B:.4f} | recall={OP_RECALL_B:.4f}",
    "",
    "## Complete TEST — Independent Model Evaluation",
    "",
    test_comparison.to_markdown(index=False),
    "",
    "## TEST patterned/unpatterned breakdown",
    "",
    group_result.to_markdown(index=False),
    "",
    "## Features",
    f"- Total feature count: **{len(feature_cols)}**",
    "- TXT/pattern-derived model features: **0**",
    "",
]

report_path = REPORTS_DIR / "AML_week3_results.md"
report_path.write_text("\n".join(lines), encoding="utf-8")
print("Report saved:", report_path)
print("\n=== DONE ===")


Report saved: /home/malnaser/Eastnets-Standard-Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/reports/week3/AML_week3_results.md

=== DONE ===
